In [ ]:
import warnings
warnings.filterwarnings("default", category=UserWarning)
warnings.filterwarnings("default", category=DeprecationWarning)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import HistGradientBoostingClassifier

sns.set_style("whitegrid")


In [ ]:
df = pd.read_csv("C:\\Coding\\Machine learning\\CSV Files\\Titanic-Dataset.csv")  # change as needed
print(df.head())
print(df.info())

In [ ]:
target = "Survived"
y = df[target]
X = df.drop(columns=[target])

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
for col in ["Age", "Fare"]:
    if col in X_train.columns:
        plt.figure(figsize=(6, 3))
        sns.histplot(X_train[col], kde=True)
        plt.title(f"{col} distribution")
        plt.show()

In [ ]:
numeric_cols = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = X_train.select_dtypes(include=["object", "category"]).columns.tolist()
numeric_transform = Pipeline(steps=[("imputer", SimpleImputer(strategy="median")),("scaler", StandardScaler())])
categorical_transform = Pipeline(steps=[("imputer", SimpleImputer(strategy="most_frequent")),("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])
preprocessor = ColumnTransformer(
    transformers=[("num", numeric_transform, numeric_cols),("cat", categorical_transform, categorical_cols)],remainder="drop")

In [ ]:
clf_lr = Pipeline([
    ("pre", preprocessor),
    ("model", LogisticRegression(max_iter=2000))])
clf_dt = Pipeline([
    ("pre", preprocessor),
    ("model", DecisionTreeClassifier())])
clf_hgb = Pipeline([("pre", preprocessor),("model", HistGradientBoostingClassifier())])

In [ ]:
clf_lr.fit(X_train, y_train)
clf_dt.fit(X_train, y_train)
clf_hgb.fit(X_train, y_train)

models = {
    "Logistic Regression": clf_lr,
    "Decision Tree": clf_dt,
    "HistGradientBoosting": clf_hgb}
for name, model in models.items():

    print(name)
    y_pred = model.predict(X_test)
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print(classification_report(y_test, y_pred))